# Test out ESF images

In [ ]:
import fsspec
import xarray as xr
import hvplot.xarray

In [ ]:
fs = fsspec.filesystem('s3', profile='esf')

In [ ]:
fs.ls('s3://cafri-share/')

In [ ]:
fs.glob('s3://cafri-share/landsat_ensemble_agb_2.0.0_????')

In [ ]:
flist = fs.glob('s3://cafri-share/landsat_ensemble_agb_2.0.0_1990/*.tiff')

In [ ]:
len(flist)

In [ ]:
flist[210]

In [ ]:
ds = xr.open_dataset(fs.open(flist[210]), engine='rasterio').squeeze('band', drop=True)

In [ ]:
ds

In [ ]:
ds['band_data'].shape

In [ ]:
ds['band_data'].hvplot.image(x='x', y='y', rasterize=True, crs='EPSG:5070', tiles='OSM', alpha=0.5, 
                             cmap='YlOrBr', title='Carbon (Mg/ha)')

## Create a VRT Mosiac of a specific year

In [ ]:
import os
import boto3
from osgeo import gdal
from botocore.exceptions import NoCredentialsError, ClientError

def create_authenticated_vrt(s3_urls, output_vrt_path, **kwargs):
    """
    Create VRT with comprehensive AWS authentication handling
    
    Parameters:
    - s3_urls: List of S3 URLs
    - output_vrt_path: Path for output VRT file
    - aws_access_key: Optional AWS access key
    - aws_secret_key: Optional AWS secret key
    - aws_session_token: Optional AWS session token
    - profile_name: Optional AWS profile name
    - region: AWS region (default: us-east-1)
    """
    
    # Method 1: Try explicit credentials first
    if kwargs.get('aws_access_key') and kwargs.get('aws_secret_key'):
        os.environ['AWS_ACCESS_KEY_ID'] = kwargs['aws_access_key']
        os.environ['AWS_SECRET_ACCESS_KEY'] = kwargs['aws_secret_key']
        if kwargs.get('aws_session_token'):
            os.environ['AWS_SESSION_TOKEN'] = kwargs['aws_session_token']
    
    # Method 2: Try boto3 session with profile
    elif kwargs.get('profile_name'):
        try:
            session = boto3.Session(profile_name=kwargs['profile_name'])
            credentials = session.get_credentials()
            if credentials:
                os.environ['AWS_ACCESS_KEY_ID'] = credentials.access_key
                os.environ['AWS_SECRET_ACCESS_KEY'] = credentials.secret_key
                if credentials.token:
                    os.environ['AWS_SESSION_TOKEN'] = credentials.token
        except Exception as e:
            print(f"Warning: Could not load profile {kwargs['profile_name']}: {e}")
    
    # Set region
    region = kwargs.get('region', 'us-east-1')
    os.environ['AWS_DEFAULT_REGION'] = region
    
    # Configure GDAL for S3
    gdal.SetConfigOption('AWS_DEFAULT_REGION', region)
    gdal.SetConfigOption('AWS_VIRTUAL_HOSTING', 'YES')
    gdal.SetConfigOption('AWS_HTTPS', 'YES')
    
    # Test access to first file
    test_url = f"/vsis3/{s3_urls[0].replace('s3://', '')}"
    test_ds = gdal.Open(test_url)
    if test_ds is None:
        raise RuntimeError(f"Cannot access {s3_urls[0]}. Check AWS credentials and permissions.")
    test_ds = None
    
    # Create VRT
    vsi_paths = [f"/vsis3/{url.replace('s3://', '')}" for url in s3_urls]
    
    vrt_options = gdal.BuildVRTOptions(
        resolution=kwargs.get('resolution', 'highest'),
        resampleAlg=kwargs.get('resample', 'nearest'),
        srcNodata=kwargs.get('src_nodata'),
        VRTNodata=kwargs.get('vrt_nodata')
    )
    
    vrt_ds = gdal.BuildVRT(output_vrt_path, vsi_paths, options=vrt_options)
    vrt_ds = None
    
    print(f"VRT created successfully: {output_vrt_path}")


In [ ]:
%%time
#for year in range(1990,2024):
for year in range(1992,1993):
    flist = fs.glob(f's3://cafri-share/landsat_ensemble_agb_2.0.0_{year}/*.tiff')
    print(year)
    s3_urls = [f's3://{f}' for f in flist]
    create_authenticated_vrt(s3_urls, f'agb_{year}.vrt', profile_name='esf')

In [ ]:
ds = xr.open_dataset(f'agb_{year}.vrt', engine='rasterio').squeeze('band', drop=True)

In [ ]:
ds

In [ ]:
ds['band_data'].nbytes/1e9

In [ ]:
%%time
da = ds['band_data'][1000:4000,12000:16000].load()d

In [ ]:
da.hvplot(x='x', y='y', rasterize=True, crs='EPSG:5070', tiles='OSM', alpha=0.5, cmap='YlOrBr', title=f'Carbon in {year} (Mg/ha)')

In [ ]:
da.encoding